# ST-OMR Synthetic Curriculum v1 — Google Colab

This notebook builds the frozen **synthetic-only** curriculum corpus. It does not load checkpoints, train a model, read real/user data, or access ScoreMosaic.

Safety model: clone the public repository, checkout one exact verified source commit, install only the Stage 1–6 rendering dependencies, validate the frozen 512-family plan, build on Colab local disk, then copy only the verified archive/evidence bundle to Google Drive.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
SOURCE_SHA = 'adc8139539d3c8cd6a2e3ee4ce4de6db4dcfeb90'
REPO_DIR = Path('/content/st-omr-training')
OUTPUT_DIR = Path('/content/st-omr-synthetic-curriculum-v1')
DRIVE_MOUNT = Path('/content/drive')
DRIVE_EXPORT_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ST-OMR-SYNTHETIC'
EXPECTED_PACKAGES = {
    'lxml': '6.1.1',
    'verovio': '6.2.1',
    'CairoSVG': '2.8.2',
    'Pillow': '12.3.0',
}

if REPO_DIR.exists() or OUTPUT_DIR.exists():
    raise RuntimeError('Fresh Colab runtime required: repository/output path already exists')
print('Safety preflight: fresh local paths OK')


In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive export root:', DRIVE_EXPORT_ROOT)


In [ ]:
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', SOURCE_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if head != SOURCE_SHA:
    raise RuntimeError(f'Repository SHA mismatch: expected {SOURCE_SHA}, got {head}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
print('Exact source checkout:', head)


In [ ]:
from importlib import metadata
import cairocffi

observed_packages = {}
for package, expected in EXPECTED_PACKAGES.items():
    actual = metadata.version(package)
    observed_packages[package] = actual
    if actual != expected:
        raise RuntimeError(f'{package}: expected {expected}, got {actual}')

runtime = {
    'python': platform.python_version(),
    'platform_system': platform.system(),
    'platform_machine': platform.machine(),
    'platform_release': platform.release(),
    'cairo_runtime': str(cairocffi.cairo_version_string()),
    'packages': observed_packages,
}
print(json.dumps(runtime, indent=2, sort_keys=True))


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from st_omr_training.synthetic_curriculum import (
    SYNTHETIC_CURRICULUM_CONFIG_FINGERPRINT,
    SYNTHETIC_CURRICULUM_PROFILE_VERSION,
    build_and_persist_synthetic_curriculum,
    curriculum_plan_summary,
)

plan = curriculum_plan_summary()
if plan['family_count'] != 512:
    raise RuntimeError('Frozen family count drifted')
if plan['family_split_counts'] != {'test': 51, 'train': 410, 'validation': 51}:
    raise RuntimeError('Frozen family split plan drifted')
if any(count != 64 for count in plan['family_profile_counts'].values()):
    raise RuntimeError('Frozen family profile balance drifted')
if plan['config_fingerprint'] != SYNTHETIC_CURRICULUM_CONFIG_FINGERPRINT:
    raise RuntimeError('Frozen config fingerprint drifted')
print(json.dumps(plan, indent=2, sort_keys=True))


In [ ]:
progress_path = Path('/content/st-omr-synthetic-progress.jsonl')

def progress(event):
    line = json.dumps(event, sort_keys=True, separators=(',', ':'), allow_nan=False)
    with progress_path.open('a', encoding='utf-8') as handle:
        handle.write(line + '\n')
    if event.get('event') == 'dataset_family_completed':
        done = int(event.get('families_completed', 0))
        total = int(event.get('families_total', 0))
        if done == 1 or done == total or done % 16 == 0:
            print(f'families {done}/{total}; samples={event.get("samples_built")}')

build = build_and_persist_synthetic_curriculum(OUTPUT_DIR, progress=progress)
if build.config_fingerprint != SYNTHETIC_CURRICULUM_CONFIG_FINGERPRINT:
    raise RuntimeError('Completed build fingerprint mismatch')

persisted_manifest = OUTPUT_DIR / 'manifest.json'
persisted_manifest_sha = hashlib.sha256(persisted_manifest.read_bytes()).hexdigest()
if persisted_manifest_sha != build.manifest_sha256:
    raise RuntimeError('Persisted manifest SHA-256 mismatch')
if len(list((OUTPUT_DIR / 'images').glob('*.png'))) != len(build.images):
    raise RuntimeError('Persisted image count mismatch')
if len(list((OUTPUT_DIR / 'targets').glob('*.musicxml'))) != len(build.targets):
    raise RuntimeError('Persisted target count mismatch')

build_summary = {
    'profile_version': SYNTHETIC_CURRICULUM_PROFILE_VERSION,
    'config_fingerprint': build.config_fingerprint,
    'build_id': build.build_id,
    'manifest_sha256': build.manifest_sha256,
    'sample_count': len(build.manifest.samples),
    'target_count': len(build.targets),
    'image_count': len(build.images),
}
print(json.dumps(build_summary, indent=2, sort_keys=True))


In [ ]:
archive = Path('/content') / f'st-omr-synthetic-curriculum-v1-{build.build_id[:16]}.tar.gz'
if archive.exists():
    raise FileExistsError(archive)
archive_command = (
    "tar --sort=name --mtime='@0' --owner=0 --group=0 --numeric-owner "
    f"-C {OUTPUT_DIR.parent} -cf - {OUTPUT_DIR.name} | gzip -n -9 > {archive}"
)
subprocess.run(['bash', '-lc', archive_command], check=True)
transport_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()

evidence = {
    'schema_version': 'st-omr-colab-synthetic-export-v1',
    'source_repository': REPO_URL,
    'source_commit': SOURCE_SHA,
    'runtime': runtime,
    'plan': plan,
    'build': build_summary,
    'transport_archive': archive.name,
    'transport_sha256': transport_sha256,
}
evidence_path = Path('/content') / f'st-omr-synthetic-curriculum-v1-{build.build_id[:16]}.evidence.json'
evidence_path.write_text(json.dumps(evidence, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n', encoding='utf-8')

destination = DRIVE_EXPORT_ROOT / build.build_id
if destination.exists():
    raise FileExistsError(f'Drive export already exists: {destination}')
destination.mkdir(parents=False, exist_ok=False)
for source in (archive, evidence_path, OUTPUT_DIR / 'manifest.sha256', OUTPUT_DIR / 'build.json'):
    shutil.copy2(source, destination / source.name)

drive_archive = destination / archive.name
if hashlib.sha256(drive_archive.read_bytes()).hexdigest() != transport_sha256:
    raise RuntimeError('Drive transport archive hash mismatch after copy')
print('EXPORT PASS')
print('Drive destination:', destination)
print('build_id:', build.build_id)
print('manifest_sha256:', build.manifest_sha256)
print('transport_sha256:', transport_sha256)


## Completion gate

Treat the corpus as generated only if the last cell prints `EXPORT PASS` and records all three identities: `build_id`, `manifest_sha256`, and `transport_sha256`. The archive hash is transport integrity only; the dataset identity remains the build ID + manifest/config identities. Do not start training from this notebook.
